# Practical Statistics for Data Scientists (Python)
# Chapter 5 — Classification

> **Author:** Satrio Arjuna Putra  
> **Reference:** *Practical Statistics for Data Scientists* — Bruce, Bruce & Gedeck (O'Reilly)

---

## 📋 Chapter Summary

Chapter 5 introduces **classification** — supervised learning where the target variable is a *category* (e.g., loan default vs paid off).

| Topic | Key Concepts |
|-------|-------------|
| **Naive Bayes** | Probabilistic classifier; assumes feature independence |
| **Discriminant Analysis (LDA/QDA)** | Decision boundary using linear/quadratic separators |
| **Logistic Regression** | Predicts probability using the logit/sigmoid function |
| **Evaluating Classification** | Confusion matrix, precision, recall, ROC-AUC |
| **Decision Trees** | Recursive binary partitioning; interpretable |
| **Bagging & Random Forests** | Ensemble of trees; reduces variance |
| **Boosting (XGBoost)** | Sequential ensemble; reduces bias |
| **Regularization** | Prevents overfitting in boosting |

---


## 🔧 Setup & Imports

In [ ]:
%matplotlib inline

import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from itertools import product

from sklearn.naive_bayes import MultinomialNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_auc_score, roc_curve,
                              precision_score, recall_score)
from sklearn.model_selection import cross_val_score
import xgboost as xgb

DATA_BASE = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/"
LOAN3000    = DATA_BASE + "loan3000.csv"
LOAN_DATA   = DATA_BASE + "loan_data.csv.gz"
FULL_TRAIN  = DATA_BASE + "full_train_set.csv.gz"

np.random.seed(42)
print("Libraries loaded!")


In [ ]:
# Load loan data
loan_data = pd.read_csv(LOAN_DATA)
loan_data['outcome'] = loan_data['outcome'].astype('category')
loan_data['purpose_'] = loan_data['purpose_'].astype('category')
loan_data['home_'] = loan_data['home_'].astype('category')
print("Loan data shape:", loan_data.shape)
print(loan_data.head())


---

## 1. Naive Bayes Classifier

### 📚 Theory

**Naive Bayes** applies Bayes' theorem with the *naive* assumption that all features are conditionally independent:

$$P(Y=k \mid X_1, \ldots, X_p) \propto P(Y=k) \prod_{j=1}^{p} P(X_j \mid Y=k)$$

**Bayes' Theorem:**
$$P(Y \mid X) = \frac{P(X \mid Y) \cdot P(Y)}{P(X)}$$

| Term | Name | Meaning |
|------|------|---------|
| $P(Y \mid X)$ | Posterior | Probability of class given features |
| $P(X \mid Y)$ | Likelihood | How likely features given class |
| $P(Y)$ | Prior | Prior probability of each class |

**Strengths:** Fast, works well with text data, handles many features.  
**Weakness:** Independence assumption is rarely true.

> **Key Insight:** Despite its "naive" assumption, Naive Bayes often performs surprisingly well in practice, especially in text classification (spam filters, sentiment analysis).


In [ ]:
# Naive Bayes with categorical features (binned borrower_score)
loan_data['borrower_score_bin'] = pd.cut(loan_data['borrower_score'], bins=10, labels=False)
loan_data['payment_inc_ratio_bin'] = pd.cut(loan_data['payment_inc_ratio'], bins=10, labels=False)

features_nb = ['borrower_score_bin', 'payment_inc_ratio_bin']
X_nb = loan_data[features_nb].fillna(0)
y_nb = (loan_data['outcome'] == 'default').astype(int)

nb_model = MultinomialNB()
nb_model.fit(X_nb, y_nb)

cv_scores = cross_val_score(nb_model, X_nb, y_nb, cv=5, scoring='accuracy')
print(f"Naive Bayes 5-Fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")


---

## 2. Discriminant Analysis (LDA)

### 📚 Theory

**Linear Discriminant Analysis (LDA)** finds a linear combination of features that best separates classes by:
1. Maximising the ratio of between-class variance to within-class variance
2. Assuming features follow a multivariate normal distribution per class

**Decision boundary:** A hyperplane equidistant from the class centroids in transformed space.

**QDA (Quadratic DA):** Relaxes the equal-covariance assumption, giving curved (quadratic) boundaries.

| Method | Boundary | Covariance |
|--------|----------|------------|
| LDA | Linear | Equal across classes |
| QDA | Quadratic | Different per class |

> **Key Insight:** LDA reduces dimensionality and provides interpretable linear boundaries. It works well when the normality and equal-variance assumptions hold.


In [ ]:
# LDA on loan3000 dataset
loan3000 = pd.read_csv(LOAN3000)
loan3000['outcome'] = loan3000['outcome'].astype('category')

predictors = ['borrower_score', 'payment_inc_ratio']
X_lda = loan3000[predictors]
y_lda = loan3000['outcome']

lda = LinearDiscriminantAnalysis()
lda.fit(X_lda, y_lda)

pred_proba = pd.DataFrame(lda.predict_proba(X_lda), columns=lda.classes_)
print("LDA class means:
", pd.DataFrame(lda.means_, columns=predictors, index=lda.classes_))
print("
Sample predictions:")
print(pred_proba.head())


In [ ]:
# Visualise LDA decision boundary
center = np.mean(lda.means_, axis=0)
slope = -lda.scalings_[0] / lda.scalings_[1]
intercept = center[1] - center[0] * slope

fig, ax = plt.subplots(figsize=(7, 5))

colors = {'default': '#d62728', 'paid off': '#1f77b4'}
for outcome, grp in loan3000.groupby('outcome'):
    ax.scatter(grp['borrower_score'], grp['payment_inc_ratio'],
               c=colors[outcome], alpha=0.4, s=15, label=outcome)

x_vals = np.array([loan3000['borrower_score'].min(), loan3000['borrower_score'].max()])
ax.plot(x_vals, slope * x_vals + intercept, 'k--', linewidth=2, label='LDA Boundary')

ax.set_xlabel('Borrower Score')
ax.set_ylabel('Payment-to-Income Ratio')
ax.set_title('LDA Decision Boundary on Loan Data')
ax.legend()
plt.tight_layout()
plt.show()


---

## 3. Logistic Regression

### 📚 Theory

**Logistic Regression** models the probability of a binary outcome using the **sigmoid (logistic) function**:

$$P(Y=1 \mid X) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 X_1 + \ldots)}} = \sigma(\mathbf{X}\boldsymbol{\beta})$$

**Logit (log-odds):**
$$\log \frac{p}{1-p} = \beta_0 + \beta_1 X_1 + \ldots + \beta_p X_p$$

- Coefficients represent change in **log-odds** per unit increase in X  
- $e^{\beta_j}$ = **odds ratio** for a 1-unit increase in $X_j$  
- Decision boundary: $P \geq 0.5$ → class 1

> **Key Insight:** Logistic regression is linear in the log-odds space but non-linear in probability space. It is robust, interpretable, and a strong baseline for binary classification.


In [ ]:
# Logistic regression on loan3000
from sklearn.preprocessing import StandardScaler

X = loan3000[['borrower_score', 'payment_inc_ratio']]
y = (loan3000['outcome'] == 'default').astype(int)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_scaled, y)

print("Coefficients:")
for feat, coef in zip(predictors, lr.coef_[0]):
    print(f"  {feat}: {coef:.4f} (odds ratio: {np.exp(coef):.4f})")
print(f"Intercept: {lr.intercept_[0]:.4f}")

acc = lr.score(X_scaled, y)
print(f"
Training Accuracy: {acc:.4f}")


In [ ]:
# Sigmoid function demonstration
p = np.linspace(0.01, 0.99, 200)
logit = np.log(p / (1 - p))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(p, logit)
axes[0].axhline(0, color='grey', linestyle='--')
axes[0].axvline(0.5, color='grey', linestyle='--')
axes[0].set_xlabel('Probability p')
axes[0].set_ylabel('Log-Odds (logit)')
axes[0].set_title('Logit Function')

z = np.linspace(-6, 6, 200)
sigmoid = 1 / (1 + np.exp(-z))
axes[1].plot(z, sigmoid)
axes[1].axhline(0.5, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('z = Xβ')
axes[1].set_ylabel('P(Y=1)')
axes[1].set_title('Sigmoid Function')

plt.tight_layout()
plt.show()


---

## 4. Evaluating Classifiers

### 📚 Theory

A classifier produces a **confusion matrix**:

|  | Predicted Positive | Predicted Negative |
|--|--|--|
| **Actual Positive** | TP | FN |
| **Actual Negative** | FP | TN |

**Key metrics:**

| Metric | Formula | When to use |
|--------|---------|-------------|
| **Accuracy** | (TP+TN)/Total | Balanced classes |
| **Precision** | TP/(TP+FP) | Cost of FP is high |
| **Recall (Sensitivity)** | TP/(TP+FN) | Cost of FN is high |
| **F1-Score** | 2·P·R/(P+R) | Balance P and R |
| **AUC-ROC** | Area under ROC curve | Overall discrimination |

**ROC Curve:** Plots TPR (Recall) vs FPR at different thresholds.

> **Key Insight:** Accuracy is misleading with imbalanced classes (e.g., 95% negative = 95% accuracy by predicting all negative). Always use Precision, Recall, F1, or AUC.


In [ ]:
# Full evaluation on logistic regression
from sklearn.model_selection import train_test_split

loan_full = pd.read_csv(FULL_TRAIN)
features = ['loan_amnt', 'term', 'annual_inc', 'dti', 'payment_inc_ratio',
            'revol_bal', 'revol_util', 'purpose_', 'home_', 'emp_length', 'borrower_score']

# Encode categoricals
loan_enc = pd.get_dummies(loan_full[features + ['outcome']], drop_first=True)
y = (loan_full['outcome'] == 'default').astype(int)
X = loan_enc.drop(columns=[c for c in loan_enc.columns if 'outcome' in c], errors='ignore')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

lr_full = LogisticRegression(max_iter=1000, C=1.0)
lr_full.fit(X_train_s, y_train)

y_pred = lr_full.predict(X_test_s)
y_prob = lr_full.predict_proba(X_test_s)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['paid off', 'default']))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")


In [ ]:
# Confusion matrix heatmap
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['paid off', 'default'],
            yticklabels=['paid off', 'default'])
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.show()


In [ ]:
# ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'Logistic Regression (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier (AUC = 0.5)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.show()


---

## 5. Decision Trees

### 📚 Theory

A **decision tree** recursively splits data into groups using binary rules on features to minimise **impurity**.

**Impurity measures:**
- **Gini impurity:** $G = 1 - \sum_k p_k^2$
- **Entropy:** $H = -\sum_k p_k \log_2 p_k$

**Algorithm (CART):**
1. For each feature and threshold, compute impurity reduction
2. Choose the split that maximises impurity reduction
3. Recurse on each child node
4. Stop when max depth reached or node is pure

**Hyperparameters:** `max_depth`, `min_samples_split`, `min_samples_leaf`

> **Key Insight:** Trees are highly interpretable but tend to overfit. Limit depth (`max_depth`) to regularise. Single trees are rarely the best model but are the building block of Random Forests and Boosting.


In [ ]:
# Decision tree on loan3000
X_tree = loan3000[predictors]
y_tree = (loan3000['outcome'] == 'default').astype(int)

tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_tree, y_tree)

print("Decision Tree Rules:")
print(export_text(tree, feature_names=list(predictors)))
print(f"
Training Accuracy: {tree.score(X_tree, y_tree):.4f}")


In [ ]:
# Visualise decision tree regions
xx, yy = np.meshgrid(
    np.linspace(loan3000['borrower_score'].min(), loan3000['borrower_score'].max(), 200),
    np.linspace(loan3000['payment_inc_ratio'].min(), loan3000['payment_inc_ratio'].max(), 200)
)
Z = tree.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
for outcome, grp in loan3000.groupby('outcome'):
    c = '#d62728' if outcome == 'default' else '#1f77b4'
    ax.scatter(grp['borrower_score'], grp['payment_inc_ratio'], c=c, alpha=0.4, s=10, label=outcome)
ax.set_xlabel('Borrower Score')
ax.set_ylabel('Payment-to-Income Ratio')
ax.set_title('Decision Tree Decision Regions (max_depth=4)')
ax.legend()
plt.tight_layout()
plt.show()


---

## 6. Bagging and Random Forests

### 📚 Theory

**Bagging (Bootstrap Aggregating):** Train many trees on bootstrap samples; average predictions.

**Random Forest** adds *feature randomness*: at each split, only a random subset of $\sqrt{p}$ features is considered.

**Benefits:**
- Reduces variance (overfitting) through averaging
- Out-of-bag (OOB) samples provide a free cross-validation estimate
- Feature importance from average impurity decrease

**Feature Importance:** Measures how much each feature reduces impurity across all trees.

> **Key Insight:** Random Forests are among the most reliable "off-the-shelf" ML algorithms. They handle non-linearity, interactions, and missing patterns automatically.


In [ ]:
# Random Forest on loan3000
rf = RandomForestClassifier(n_estimators=500, oob_score=True, random_state=42)
rf.fit(X_tree, y_tree)

print(f"OOB Score: {rf.oob_score_:.4f}")
print(f"Training Accuracy: {rf.score(X_tree, y_tree):.4f}")

# Feature importance
importance_df = pd.DataFrame({
    'Feature': predictors,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=False)
print("
Feature Importances:")
print(importance_df)


In [ ]:
# Random Forest with more features (full dataset)
X_rf = loan3000[predictors]
y_rf = y_tree

rf_cv = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=42),
                         X_rf, y_rf, cv=5, scoring='accuracy')
print(f"Random Forest 5-Fold CV Accuracy: {rf_cv.mean():.4f} ± {rf_cv.std():.4f}")

# Visualise RF decision boundary
xx, yy = np.meshgrid(
    np.linspace(loan3000['borrower_score'].min(), loan3000['borrower_score'].max(), 200),
    np.linspace(loan3000['payment_inc_ratio'].min(), loan3000['payment_inc_ratio'].max(), 200)
)
Z_rf = rf.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
fig, ax = plt.subplots(figsize=(7, 5))
ax.contourf(xx, yy, Z_rf, alpha=0.3, cmap='RdBu')
for outcome, grp in loan3000.groupby('outcome'):
    c = '#d62728' if outcome == 'default' else '#1f77b4'
    ax.scatter(grp['borrower_score'], grp['payment_inc_ratio'], c=c, alpha=0.4, s=10, label=outcome)
ax.set_xlabel('Borrower Score')
ax.set_ylabel('Payment-to-Income Ratio')
ax.set_title('Random Forest Decision Regions (500 trees)')
ax.legend()
plt.tight_layout()
plt.show()


---

## 7. Boosting (XGBoost)

### 📚 Theory

**Boosting** builds an ensemble *sequentially* — each new tree corrects the errors of previous trees.

**Gradient Boosting:**
$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

Where $h_m$ is fitted to the *residuals* (negative gradient of loss) from $F_{m-1}$.

**XGBoost** adds:
- **Regularization (L1/L2)** to penalise tree complexity
- **Column subsampling** for variance reduction
- Efficient parallel tree building

**Key hyperparameters:**

| Parameter | Effect |
|-----------|--------|
| `n_estimators` | Number of trees (more = more potential overfit) |
| `learning_rate (η)` | Shrinkage; smaller = slower but safer |
| `max_depth` | Tree depth; larger = more complex |
| `subsample` | Row fraction per tree |
| `colsample_bytree` | Feature fraction per tree |

> **Key Insight:** XGBoost consistently wins tabular ML competitions. Always tune `n_estimators` + `learning_rate` together; lower learning rate + more trees often wins.


In [ ]:
# XGBoost on loan3000
y_xgb = pd.Series([1 if o == 'default' else 0 for o in loan3000['outcome']])

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)
xgb_model.fit(X_tree, y_xgb)

print(f"XGBoost Training Accuracy: {xgb_model.score(X_tree, y_xgb):.4f}")

xgb_cv = cross_val_score(xgb_model, X_tree, y_xgb, cv=5, scoring='accuracy')
print(f"XGBoost 5-Fold CV Accuracy: {xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}")


In [ ]:
# Feature importance from XGBoost
xgb.plot_importance(xgb_model, max_num_features=5, importance_type='gain')
plt.title('XGBoost Feature Importance (by Gain)')
plt.tight_layout()
plt.show()


---

## ✅ Chapter 5 Key Takeaways

1. **Naive Bayes:** Fast and effective, especially for high-dimensional data; independence assumption is limiting.
2. **LDA:** Linear boundary; requires normality and equal covariance across classes.
3. **Logistic Regression:** Models log-odds linearly; coefficients are interpretable as odds ratios.
4. **Accuracy alone is misleading** — always use Precision, Recall, F1, AUC for imbalanced data.
5. **Decision Trees:** Interpretable but prone to overfitting; regularise with `max_depth`.
6. **Random Forests:** Powerful ensemble; OOB score provides free validation; feature importance ranks predictors.
7. **XGBoost:** State-of-the-art for tabular data; requires careful tuning of `n_estimators` and `learning_rate`.
8. **Regularization** is crucial for boosting to prevent overfitting.

---
*Reference: Chapter 5, Practical Statistics for Data Scientists (O'Reilly)*
